# 🧠 Notebook 1: Data Loading & Exploration

## Brain Tumor Classification using Differential Equation Modeling

**Dataset:** BraTS 2020 Training Data  
**Source:** [Kaggle - awsaf49/brats2020-training-data](https://www.kaggle.com/datasets/awsaf49/brats2020-training-data)

---

### Objective
This notebook loads the BraTS 2020 brain tumor MRI dataset, explores its structure, and visualizes the multi-modal MRI scans along with segmentation masks.

### Pipeline Overview
```
MRI Brain Image → Grayscale Conversion → Resize to 10×10 → Row-wise Mean → 1D Signal
    → Differential Modeling (DE/IDE with Euler/RK4) → CNN Classification → Benign/Malignant
```

### Dataset Description
- **369 subjects** with multi-institutional pre-operative MRI scans
- **4 MRI modalities:** T1, T1ce (contrast-enhanced), T2, FLAIR
- **Segmentation labels:** ET (label 4), ED (label 2), NCR/NET (label 1)
- **Volume dimensions:** 240 × 240 × 155 voxels per modality
- **Grades:** HGG (High Grade Glioma) → **Malignant**, LGG (Low Grade Glioma) → **Benign**

## 1. Install & Import Dependencies

In [ ]:
# Install required packages (uncomment if needed)
# !pip install nibabel h5py scikit-image tqdm seaborn

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import h5py
import warnings
from glob import glob
from tqdm import tqdm
from collections import Counter

# Try importing nibabel for NIfTI support
try:
    import nibabel as nib
    HAS_NIBABEL = True
    print("✅ nibabel available - NIfTI format supported")
except ImportError:
    HAS_NIBABEL = False
    print("⚠️ nibabel not found. Install with: pip install nibabel")

warnings.filterwarnings('ignore')
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12

# Reproducibility
np.random.seed(42)

print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 2. Configuration & Data Paths

Set the `DATA_DIR` variable below to point to your BraTS2020 dataset location.

### Expected Directory Structure
```
BraTS2020_TrainingData/
├── MICCAI_BraTS2020_TrainingData/
│   ├── BraTS20_Training_001/
│   │   ├── BraTS20_Training_001_flair.nii
│   │   ├── BraTS20_Training_001_t1.nii
│   │   ├── BraTS20_Training_001_t1ce.nii
│   │   ├── BraTS20_Training_001_t2.nii
│   │   └── BraTS20_Training_001_seg.nii
│   ├── BraTS20_Training_002/
│   │   └── ...
│   ├── name_mapping.csv
│   └── survival_info.csv
```

In [ ]:
# ============================================================
#  ⚙️  CONFIGURATION - Update these paths for your environment
# ============================================================

# --- Option 1: Kaggle Environment ---
# DATA_DIR = '/kaggle/input/brats2020-training-data/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'

# --- Option 2: Local Environment ---
DATA_DIR = './data/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'

# Output directory for processed data (shared across notebooks)
OUTPUT_DIR = './processed_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Number of subjects to load for exploration (set to None for all)
MAX_SUBJECTS = 50  # Use a subset for quick exploration

print(f"Data directory : {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Data dir exists : {os.path.exists(DATA_DIR)}")

## 3. Data Loading Utilities

We define helper functions to load BraTS data from either **NIfTI** (`.nii` / `.nii.gz`) or **HDF5** (`.h5`) format.

In [ ]:
def load_nifti_volume(filepath):
    """Load a NIfTI (.nii / .nii.gz) volume and return as numpy array."""
    img = nib.load(filepath)
    return img.get_fdata().astype(np.float32)


def load_subject_nifti(subject_dir):
    """
    Load all 4 MRI modalities + segmentation mask for one subject.
    Returns dict with keys: 'flair', 't1', 't1ce', 't2', 'seg'
    Each value is a numpy array of shape (240, 240, 155).
    """
    subject_id = os.path.basename(subject_dir)
    modalities = {}
    
    for mod in ['flair', 't1', 't1ce', 't2', 'seg']:
        # Try both .nii and .nii.gz extensions
        nii_path = os.path.join(subject_dir, f'{subject_id}_{mod}.nii')
        nii_gz_path = os.path.join(subject_dir, f'{subject_id}_{mod}.nii.gz')
        
        if os.path.exists(nii_path):
            modalities[mod] = load_nifti_volume(nii_path)
        elif os.path.exists(nii_gz_path):
            modalities[mod] = load_nifti_volume(nii_gz_path)
        else:
            print(f"  ⚠️ Missing {mod} for {subject_id}")
            modalities[mod] = None
    
    return modalities


def load_h5_slice(h5_path):
    """
    Load a single slice from HDF5 format.
    Returns image (H, W, 4) and mask (H, W, 3).
    """
    with h5py.File(h5_path, 'r') as f:
        image = f['image'][:]
        mask = f['mask'][:]
    return image, mask


def get_subject_list(data_dir):
    """Get sorted list of subject directories."""
    pattern = os.path.join(data_dir, 'BraTS20_Training_*')
    dirs = sorted([d for d in glob(pattern) if os.path.isdir(d)])
    return dirs


def load_name_mapping(data_dir):
    """
    Load name_mapping.csv to get HGG/LGG grade information.
    Returns DataFrame with subject ID and grade.
    """
    csv_path = os.path.join(data_dir, 'name_mapping.csv')
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        print(f"✅ Loaded name_mapping.csv with {len(df)} entries")
        print(f"   Columns: {list(df.columns)}")
        return df
    else:
        print("⚠️ name_mapping.csv not found. Will infer labels from folder structure.")
        return None


print("✅ Data loading utilities defined.")

## 4. Discover and Catalog the Dataset

Scan the data directory to understand what's available.

In [ ]:
# Check if real data is available
USE_SYNTHETIC = False

if os.path.exists(DATA_DIR):
    subject_dirs = get_subject_list(DATA_DIR)
    if len(subject_dirs) > 0:
        print(f"✅ Found {len(subject_dirs)} subjects in {DATA_DIR}")
        print(f"   First: {os.path.basename(subject_dirs[0])}")
        print(f"   Last : {os.path.basename(subject_dirs[-1])}")
        
        # Show contents of first subject
        first_files = os.listdir(subject_dirs[0])
        print(f"\n   Files in first subject directory:")
        for f in sorted(first_files):
            fpath = os.path.join(subject_dirs[0], f)
            size_mb = os.path.getsize(fpath) / (1024*1024)
            print(f"     {f} ({size_mb:.1f} MB)")
    else:
        print(f"⚠️ No subject directories found in {DATA_DIR}")
        USE_SYNTHETIC = True
else:
    print(f"❌ Data directory not found: {DATA_DIR}")
    print("   Using SYNTHETIC data for demonstration.")
    print("   ➡️ Download from: https://www.kaggle.com/datasets/awsaf49/brats2020-training-data")
    USE_SYNTHETIC = True

In [ ]:
# Generate synthetic demo data if real data is not available
def generate_synthetic_data(n_subjects=50, volume_shape=(240, 240, 155)):
    """
    Generate synthetic BraTS-like data for demonstration purposes.
    Creates realistic-looking brain MRI slices with simulated tumors.
    """
    print("\n🔧 Generating synthetic BraTS-like data...")
    H, W, D = volume_shape
    subjects = []
    labels = []  # 0 = Benign (LGG), 1 = Malignant (HGG)
    
    for i in tqdm(range(n_subjects), desc="Generating subjects"):
        # Assign grade: ~75% HGG, ~25% LGG (matching real distribution)
        is_hgg = np.random.random() < 0.75
        grade = 1 if is_hgg else 0
        labels.append(grade)
        
        # Generate brain-like base (elliptical shape with intensity variation)
        y, x = np.ogrid[-1:1:H*1j, -1:1:W*1j]
        brain_mask = (x**2 / 0.7**2 + y**2 / 0.9**2) < 1
        
        # Create 4 modalities with different intensity profiles
        base_intensity = np.random.uniform(0.3, 0.7)
        modalities = {}
        for mod_name, mod_scale in [('flair', 1.0), ('t1', 0.8), ('t1ce', 0.9), ('t2', 1.1)]:
            mod_slice = np.zeros((H, W), dtype=np.float32)
            mod_slice[brain_mask] = base_intensity * mod_scale + np.random.normal(0, 0.05, brain_mask.sum())
            modalities[mod_name] = np.clip(mod_slice, 0, 1)
        
        # Add tumor region
        seg = np.zeros((H, W), dtype=np.float32)
        tumor_cx = np.random.randint(80, 160)
        tumor_cy = np.random.randint(80, 160)
        tumor_r = np.random.randint(15, 40) if is_hgg else np.random.randint(8, 20)
        
        tumor_mask = ((x * W/2 + W/2 - tumor_cx)**2 + (y * H/2 + H/2 - tumor_cy)**2) < tumor_r**2
        tumor_mask = tumor_mask & brain_mask
        
        # Enhancement for HGG tumors (brighter in T1ce)
        if is_hgg:
            modalities['t1ce'][tumor_mask] *= 1.5
            modalities['flair'][tumor_mask] *= 1.3
            seg[tumor_mask] = 4  # ET label
        else:
            modalities['flair'][tumor_mask] *= 1.2
            seg[tumor_mask] = 1  # NCR label
        
        # Add edema around tumor
        edema_r = tumor_r + np.random.randint(5, 15)
        edema_mask = ((x * W/2 + W/2 - tumor_cx)**2 + (y * H/2 + H/2 - tumor_cy)**2) < edema_r**2
        edema_mask = edema_mask & brain_mask & ~tumor_mask
        modalities['flair'][edema_mask] *= 1.15
        seg[edema_mask] = 2  # ED label
        
        modalities['seg'] = seg
        subjects.append(modalities)
    
    print(f"✅ Generated {n_subjects} synthetic subjects")
    print(f"   HGG (Malignant): {sum(labels)}, LGG (Benign): {n_subjects - sum(labels)}")
    return subjects, np.array(labels)


if USE_SYNTHETIC:
    synthetic_subjects, synthetic_labels = generate_synthetic_data(n_subjects=50)
    print("\nℹ️ Synthetic data will be used for all visualizations below.")

## 5. Load Name Mapping (Grade Information)

The `name_mapping.csv` file contains the mapping between subject IDs and their tumor grade:
- **HGG** (High Grade Glioma) → **Malignant** (label = 1)
- **LGG** (Low Grade Glioma) → **Benign** (label = 0)

In [ ]:
if not USE_SYNTHETIC:
    # Load name mapping for grade info
    name_mapping_df = load_name_mapping(DATA_DIR)
    
    if name_mapping_df is not None:
        display(name_mapping_df.head(10))
        
        # Extract grade distribution
        if 'Grade' in name_mapping_df.columns:
            grade_col = 'Grade'
        elif 'grade' in name_mapping_df.columns:
            grade_col = 'grade'
        else:
            # Try to find the grade column
            grade_col = None
            for col in name_mapping_df.columns:
                unique_vals = name_mapping_df[col].unique()
                if set(unique_vals).issubset({'HGG', 'LGG'}):
                    grade_col = col
                    break
        
        if grade_col:
            grade_counts = name_mapping_df[grade_col].value_counts()
            print(f"\n📊 Grade Distribution:")
            print(f"   HGG (Malignant): {grade_counts.get('HGG', 0)}")
            print(f"   LGG (Benign)   : {grade_counts.get('LGG', 0)}")
        else:
            print("⚠️ Could not identify grade column.")
    
    # Also check survival info
    survival_path = os.path.join(DATA_DIR, 'survival_info.csv')
    if os.path.exists(survival_path):
        survival_df = pd.read_csv(survival_path)
        print(f"\n✅ Loaded survival_info.csv with {len(survival_df)} entries")
        display(survival_df.head())
else:
    print("Using synthetic data - grade labels already assigned.")
    grade_dist = Counter(synthetic_labels)
    print(f"Malignant (HGG): {grade_dist[1]}, Benign (LGG): {grade_dist[0]}")

## 6. Visualize MRI Modalities

Each subject has **4 MRI modalities** that provide complementary information:

| Modality | Description | Best For |
|----------|-------------|----------|
| **T1** | Native T1-weighted | Anatomical structure |
| **T1ce** | Post-contrast T1-weighted | Enhancing tumor (ET) |
| **T2** | T2-weighted | Peritumoral edema (ED) |
| **FLAIR** | T2 Fluid Attenuated Inversion Recovery | Whole tumor extent |

In [ ]:
def visualize_modalities(subject_data, subject_id="Sample", slice_idx=None):
    """
    Visualize all 4 MRI modalities and segmentation mask for a single slice.
    """
    modality_names = ['FLAIR', 'T1', 'T1ce', 'T2', 'Segmentation']
    modality_keys = ['flair', 't1', 't1ce', 't2', 'seg']
    cmaps = ['gray', 'gray', 'gray', 'gray', 'nipy_spectral']
    
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    fig.suptitle(f'Subject: {subject_id}', fontsize=16, fontweight='bold', y=1.02)
    
    for idx, (name, key, cmap) in enumerate(zip(modality_names, modality_keys, cmaps)):
        data = subject_data[key]
        if data is None:
            axes[idx].text(0.5, 0.5, 'N/A', ha='center', va='center', fontsize=14)
        else:
            # Handle 3D volumes (take middle slice) vs 2D slices
            if data.ndim == 3:
                s_idx = slice_idx if slice_idx else data.shape[2] // 2
                img = data[:, :, s_idx]
            else:
                img = data
            
            axes[idx].imshow(img, cmap=cmap)
        
        axes[idx].set_title(name, fontsize=13, fontweight='bold')
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()


# Visualize a sample subject
if USE_SYNTHETIC:
    # Plot for both HGG and LGG synthetic
    hgg_idx = np.where(synthetic_labels == 1)[0][0]
    lgg_idx = np.where(synthetic_labels == 0)[0][0]
    
    print("🔴 Malignant (HGG) Sample:")
    visualize_modalities(synthetic_subjects[hgg_idx], f"Synthetic HGG #{hgg_idx}")
    
    print("\n🟢 Benign (LGG) Sample:")
    visualize_modalities(synthetic_subjects[lgg_idx], f"Synthetic LGG #{lgg_idx}")
else:
    # Load and visualize first subject
    sample_dir = subject_dirs[0]
    sample_id = os.path.basename(sample_dir)
    print(f"Loading subject: {sample_id}...")
    sample_data = load_subject_nifti(sample_dir)
    visualize_modalities(sample_data, sample_id)

## 7. Multi-Slice Visualization

Visualize multiple axial slices from a single subject to show the tumor across different depths.

In [ ]:
def visualize_multi_slice(subject_data, modality='flair', n_slices=8, subject_id="Sample"):
    """
    Visualize multiple slices from a selected modality.
    """
    data = subject_data[modality]
    seg_data = subject_data['seg']
    
    if data is None:
        print(f"No data for modality: {modality}")
        return
    
    if data.ndim == 3:
        total_slices = data.shape[2]
        # Select slices evenly spaced through the volume
        slice_indices = np.linspace(30, total_slices - 30, n_slices, dtype=int)
    else:
        # 2D data: show the same slice repeated
        slice_indices = [0] * n_slices
    
    fig, axes = plt.subplots(2, n_slices, figsize=(n_slices * 2.5, 5))
    fig.suptitle(f'{subject_id} - {modality.upper()} Modality (Multiple Slices)', 
                 fontsize=14, fontweight='bold')
    
    for idx, s_idx in enumerate(slice_indices):
        if data.ndim == 3:
            img = data[:, :, s_idx]
            seg = seg_data[:, :, s_idx] if seg_data is not None and seg_data.ndim == 3 else None
        else:
            img = data
            seg = seg_data
        
        # MRI slice
        axes[0, idx].imshow(img, cmap='gray')
        axes[0, idx].set_title(f'Slice {s_idx}', fontsize=10)
        axes[0, idx].axis('off')
        
        # Segmentation overlay
        axes[1, idx].imshow(img, cmap='gray')
        if seg is not None:
            seg_masked = np.ma.masked_where(seg == 0, seg)
            axes[1, idx].imshow(seg_masked, cmap='hot', alpha=0.5, vmin=0, vmax=4)
        axes[1, idx].set_title(f'+ Tumor', fontsize=10)
        axes[1, idx].axis('off')
    
    axes[0, 0].set_ylabel('MRI', fontsize=12, fontweight='bold')
    axes[1, 0].set_ylabel('Overlay', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()


if USE_SYNTHETIC:
    print("ℹ️ Synthetic data is 2D - showing single slice per subject.")
    # Show multiple subjects instead
    fig, axes = plt.subplots(2, 6, figsize=(18, 6))
    fig.suptitle('Sample Subjects: FLAIR Modality + Tumor Overlay', fontsize=14, fontweight='bold')
    
    for idx in range(6):
        subj = synthetic_subjects[idx]
        label_str = "HGG" if synthetic_labels[idx] == 1 else "LGG"
        
        axes[0, idx].imshow(subj['flair'], cmap='gray')
        axes[0, idx].set_title(f'#{idx} ({label_str})', fontsize=10)
        axes[0, idx].axis('off')
        
        axes[1, idx].imshow(subj['flair'], cmap='gray')
        seg_masked = np.ma.masked_where(subj['seg'] == 0, subj['seg'])
        axes[1, idx].imshow(seg_masked, cmap='hot', alpha=0.5, vmin=0, vmax=4)
        axes[1, idx].set_title('+ Tumor', fontsize=10)
        axes[1, idx].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    visualize_multi_slice(sample_data, 'flair', n_slices=8, subject_id=sample_id)

## 8. Segmentation Mask Analysis

Analyze the distribution of tumor sub-regions across the dataset.

| Label | Sub-region | Color |
|-------|-----------|-------|
| 0 | Background | Black |
| 1 | NCR/NET (Necrotic/Non-enhancing tumor) | Red |
| 2 | ED (Peritumoral Edema) | Yellow |
| 4 | ET (Enhancing Tumor) | Green |

In [ ]:
def analyze_segmentation(subject_data, subject_id="Sample"):
    """
    Analyze and visualize segmentation mask statistics.
    """
    seg = subject_data['seg']
    if seg is None:
        print("No segmentation data available.")
        return {}
    
    # Count voxels per label
    unique, counts = np.unique(seg, return_counts=True)
    label_names = {0: 'Background', 1: 'NCR/NET', 2: 'Edema (ED)', 4: 'Enhancing (ET)'}
    
    print(f"\n📊 Segmentation Statistics for {subject_id}:")
    total_voxels = seg.size
    for label, count in zip(unique, counts):
        name = label_names.get(int(label), f'Unknown ({int(label)})')
        pct = 100.0 * count / total_voxels
        print(f"   Label {int(label):d} ({name:20s}): {count:>10,d} voxels ({pct:.3f}%)")
    
    # Tumor volume (non-background)
    tumor_voxels = np.sum(seg > 0)
    print(f"\n   Total tumor voxels: {tumor_voxels:,d} ({100.0 * tumor_voxels / total_voxels:.3f}%)")
    
    return dict(zip(unique.astype(int), counts))


if USE_SYNTHETIC:
    _ = analyze_segmentation(synthetic_subjects[hgg_idx], f"Synthetic HGG #{hgg_idx}")
    _ = analyze_segmentation(synthetic_subjects[lgg_idx], f"Synthetic LGG #{lgg_idx}")
else:
    _ = analyze_segmentation(sample_data, sample_id)

## 9. Intensity Distribution Analysis

Compare the intensity distributions across different MRI modalities for brain tissue vs. tumor regions.

In [ ]:
def plot_intensity_distributions(subject_data, subject_id="Sample"):
    """
    Plot intensity distributions for each modality, separated by tissue type.
    """
    modality_keys = ['flair', 't1', 't1ce', 't2']
    modality_names = ['FLAIR', 'T1', 'T1ce', 'T2']
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    fig.suptitle(f'Intensity Distributions - {subject_id}', fontsize=14, fontweight='bold')
    
    seg = subject_data['seg']
    if seg is not None and seg.ndim == 3:
        # Use middle slice for 3D data
        slice_idx = seg.shape[2] // 2
        seg_slice = seg[:, :, slice_idx]
    else:
        seg_slice = seg
    
    for idx, (key, name) in enumerate(zip(modality_keys, modality_names)):
        data = subject_data[key]
        if data is None:
            continue
        
        if data.ndim == 3:
            data_slice = data[:, :, slice_idx]
        else:
            data_slice = data
        
        # Separate brain and tumor intensities
        brain_mask = data_slice > 0
        tumor_mask = seg_slice > 0 if seg_slice is not None else np.zeros_like(data_slice, dtype=bool)
        healthy_mask = brain_mask & ~tumor_mask
        
        brain_vals = data_slice[healthy_mask].flatten()
        tumor_vals = data_slice[tumor_mask].flatten()
        
        if len(brain_vals) > 0:
            axes[idx].hist(brain_vals, bins=50, alpha=0.6, color='steelblue', 
                          label='Healthy', density=True)
        if len(tumor_vals) > 0:
            axes[idx].hist(tumor_vals, bins=50, alpha=0.6, color='crimson', 
                          label='Tumor', density=True)
        
        axes[idx].set_title(name, fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('Intensity')
        axes[idx].set_ylabel('Density')
        axes[idx].legend(fontsize=9)
    
    plt.tight_layout()
    plt.show()


if USE_SYNTHETIC:
    plot_intensity_distributions(synthetic_subjects[hgg_idx], f"HGG #{hgg_idx}")
else:
    plot_intensity_distributions(sample_data, sample_id)

## 10. Grade Distribution Visualization

Visualize the class distribution (HGG vs LGG) which maps to our **Malignant vs Benign** classification.

In [ ]:
def plot_grade_distribution(labels, title="Tumor Grade Distribution"):
    """
    Plot the distribution of tumor grades.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    counts = Counter(labels)
    categories = ['Benign (LGG)', 'Malignant (HGG)']
    values = [counts.get(0, 0), counts.get(1, 0)]
    colors = ['#2ecc71', '#e74c3c']
    
    # Bar chart
    bars = ax1.bar(categories, values, color=colors, edgecolor='black', linewidth=1.2)
    ax1.set_ylabel('Number of Subjects', fontsize=12)
    ax1.set_title(title, fontsize=14, fontweight='bold')
    for bar, val in zip(bars, values):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
                str(val), ha='center', va='bottom', fontweight='bold', fontsize=13)
    
    # Pie chart
    ax2.pie(values, labels=categories, colors=colors, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12}, 
            wedgeprops={'edgecolor': 'black', 'linewidth': 1.2})
    ax2.set_title('Class Proportion', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Print imbalance ratio
    minority = min(values)
    majority = max(values)
    print(f"\n📊 Class Imbalance Ratio: 1:{majority/minority:.1f} (minority:majority)")
    print(f"   Total samples: {sum(values)}")


if USE_SYNTHETIC:
    plot_grade_distribution(synthetic_labels, "Synthetic Data - Grade Distribution")
else:
    if name_mapping_df is not None and grade_col:
        labels_from_csv = (name_mapping_df[grade_col] == 'HGG').astype(int).values
        plot_grade_distribution(labels_from_csv, "BraTS 2020 - Tumor Grade Distribution")
    else:
        print("Grade information not available for distribution plot.")

## 11. Build Subject-Level Label Mapping

Create a dictionary mapping each subject ID to its binary label (0=Benign, 1=Malignant) and save it for subsequent notebooks.

In [ ]:
def build_label_mapping(data_dir, name_mapping_df=None, grade_col=None):
    """
    Build a mapping: subject_id -> binary_label.
    HGG -> 1 (Malignant), LGG -> 0 (Benign).
    """
    label_map = {}
    
    if name_mapping_df is not None and grade_col is not None:
        # Use CSV mapping
        id_col = None
        for col in name_mapping_df.columns:
            if 'BraTS_2020' in col or 'subject' in col.lower():
                id_col = col
                break
        
        if id_col is None:
            id_col = name_mapping_df.columns[0]
        
        for _, row in name_mapping_df.iterrows():
            subj_id = row[id_col]
            grade = row[grade_col]
            label_map[subj_id] = 1 if grade == 'HGG' else 0
    else:
        # Fallback: check folder structure for HGG/LGG subdirectories
        for grade_folder in ['HGG', 'LGG']:
            grade_path = os.path.join(data_dir, grade_folder)
            if os.path.exists(grade_path):
                grade_label = 1 if grade_folder == 'HGG' else 0
                for subj in os.listdir(grade_path):
                    if subj.startswith('BraTS'):
                        label_map[subj] = grade_label
    
    return label_map


if not USE_SYNTHETIC:
    label_mapping = build_label_mapping(DATA_DIR, 
                                         name_mapping_df if 'name_mapping_df' in dir() else None,
                                         grade_col if 'grade_col' in dir() else None)
    print(f"✅ Built label mapping for {len(label_mapping)} subjects")
    
    # Save mapping
    label_df = pd.DataFrame(list(label_mapping.items()), columns=['subject_id', 'label'])
    label_df['grade'] = label_df['label'].map({0: 'LGG (Benign)', 1: 'HGG (Malignant)'})
    label_df.to_csv(os.path.join(OUTPUT_DIR, 'subject_labels.csv'), index=False)
    print(f"   Saved to: {os.path.join(OUTPUT_DIR, 'subject_labels.csv')}")
    display(label_df.head())
else:
    # Save synthetic labels
    label_df = pd.DataFrame({
        'subject_id': [f'Synthetic_{i:03d}' for i in range(len(synthetic_labels))],
        'label': synthetic_labels,
        'grade': ['HGG (Malignant)' if l == 1 else 'LGG (Benign)' for l in synthetic_labels]
    })
    label_df.to_csv(os.path.join(OUTPUT_DIR, 'subject_labels.csv'), index=False)
    print(f"✅ Saved label mapping to: {os.path.join(OUTPUT_DIR, 'subject_labels.csv')}")
    display(label_df.head())

## 12. Dataset Summary Statistics

Generate comprehensive summary statistics for the dataset.

In [ ]:
# Comprehensive summary
print("=" * 60)
print(" BraTS 2020 DATASET EXPLORATION SUMMARY")
print("=" * 60)

if USE_SYNTHETIC:
    n_subjects = len(synthetic_subjects)
    n_hgg = int(sum(synthetic_labels))
    n_lgg = n_subjects - n_hgg
    data_type = "SYNTHETIC (for demonstration)"
else:
    n_subjects = len(subject_dirs)
    n_hgg = sum(1 for v in label_mapping.values() if v == 1) if label_mapping else '?'
    n_lgg = sum(1 for v in label_mapping.values() if v == 0) if label_mapping else '?'
    data_type = "REAL BraTS 2020"

print(f"\n  Data Type        : {data_type}")
print(f"  Total Subjects   : {n_subjects}")
print(f"  HGG (Malignant)  : {n_hgg}")
print(f"  LGG (Benign)     : {n_lgg}")
print(f"  MRI Modalities   : FLAIR, T1, T1ce, T2")
print(f"  Volume Shape     : 240 x 240 x 155")
print(f"  Voxel Resolution : 1mm\u00b3 isotropic")
print(f"  Seg. Labels      : 0 (BG), 1 (NCR/NET), 2 (ED), 4 (ET)")
print(f"\n  Output saved to  : {OUTPUT_DIR}")
print("=" * 60)

## 13. Save Configuration for Subsequent Notebooks

Export key configuration so the remaining notebooks can pick up from here.

In [ ]:
import json

config = {
    'DATA_DIR': DATA_DIR,
    'OUTPUT_DIR': OUTPUT_DIR,
    'USE_SYNTHETIC': USE_SYNTHETIC,
    'N_SUBJECTS': n_subjects,
    'MAX_SUBJECTS': MAX_SUBJECTS,
    'VOLUME_SHAPE': [240, 240, 155],
    'MODALITIES': ['flair', 't1', 't1ce', 't2'],
    'SEG_LABELS': {0: 'Background', 1: 'NCR/NET', 2: 'Edema', 4: 'Enhancing Tumor'}
}

config_path = os.path.join(OUTPUT_DIR, 'config.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f"✅ Configuration saved to: {config_path}")
print(json.dumps(config, indent=2))

---

## ✅ Summary

In this notebook, we:

1. **Loaded** the BraTS 2020 dataset (or generated synthetic data for demonstration)
2. **Explored** the directory structure and file formats
3. **Visualized** all 4 MRI modalities (FLAIR, T1, T1ce, T2) with segmentation overlays
4. **Analyzed** tumor region statistics and intensity distributions
5. **Mapped** tumor grades (HGG/LGG) to binary labels (Malignant/Benign)
6. **Saved** configuration and label mappings for subsequent notebooks

### ➡️ Next: [Notebook 02 - Preprocessing Pipeline](./02_Preprocessing_Pipeline.ipynb)

The next notebook will implement the preprocessing steps:
- Grayscale conversion
- Resize to 10×10
- Row-wise mean → 1D signal extraction